# Usage of GC.Analysis.API for GC Analysis

In [1]:
#r "nuget: Microsoft.Diagnostics.Tracing.TraceEvent, 3.1.13"
#r "nuget: XPlot.Plotly"
#r "nuget: Microsoft.Data.Analysis"
#r "nuget: Newtonsoft.Json"

using Etlx = Microsoft.Diagnostics.Tracing.Etlx;
using Microsoft.Data.Analysis;
using Microsoft.Diagnostics.Tracing.Analysis.GC;
using Microsoft.Diagnostics.Tracing.Analysis;
using Microsoft.Diagnostics.Tracing.Parsers.Clr;
using Microsoft.Diagnostics.Tracing;
using XPlot.Plotly;

using System.IO;
using Newtonsoft.Json;

Installed Packages Microsoft.Data.Analysis, 0.22.2 Microsoft.Diagnostics.Tracing.TraceEvent, 3.1.13 Newtonsoft.Json, 13.0.3 XPlot.Plotly, 4.0.6

Loading extensions from `C:\Users\musharm\.nuget\packages\microsoft.data.analysis\0.22.2\interactive-extensions\dotnet\Microsoft.Data.Analysis.Interactive.dll`

## Building and Using The GC Analysis API

In [2]:
dotnet build -c Release "..\GC.Analysis.API"

  Determining projects to restore...
  All projects are up-to-date for restore.
  GC.Analysis.API -> C:\Users\musharm\source\repos\performance\artifacts\bin\GC.Analysis.API\Release\net8.0\GC.Analysis.API.dll

Build succeeded.
    0 Warning(s)
    0 Error(s)

Time Elapsed 00:00:01.31


In [3]:
#r "..\..\..\..\..\artifacts\bin\GC.Analysis.API\Release\net8.0\GC.Analysis.API.dll"

using GC.Analysis.API;

## No Env Analysis

In [8]:
var basePath_NoEnv = @"C:\Users\musharm\Downloads\runs2\runs2\noenv";
var basePath_8HeapCount = @"C:\Users\musharm\Downloads\runs2\runs2\GCHeapCount8";

In [ ]:
public class Comparison
{
    public GCProcessData Baseline { get; set; }
    public GCProcessData Comparand { get; set; }
}

public class DiffStatistics
{
    public double AverageTotalPauseTimeDiffPercentage { get; set; }
    public double AverageMeanHeapSizeDiffPercentage { get; set; }
    public double AverageTotalAllocDiffPercentage { get; set; }
    public double AverageGen0PauseTimeDiffPercentage { get; set; }
    public double AverageGen1PauseTimeDiffPercentage { get; set; }
    public double AverageGen2BlockingPauseTimeDiffPercentage { get; set; }
    public double AverageFirstLastTimeDiff { get; set; }
    public double AverageProcessDurationDiff { get; set; }
}

public class ComparisonManager
{
    public ComparisonManager(string basePath)
    {
        Dictionary<string, Analyzer> gcTraceData = AnalyzerManager.GetAllAnalyzers(basePath);
        Comparisons = GetCombinedGroups(gcTraceData);
        double averageTotalPauseTimeDiffPercentage     = 0;
        double averageTotalAllocDiffPercentage         = 0;
        double averageMeanHeapSizeBeforeDiffPercentage = 0;
        double averageGen0PauseTimeDiffPercentage = 0;
        double averageGen1PauseTimeDiffPercentage = 0;
        double averageGen2PauseTimeDiffPercentage = 0;
        double averageFirstLastTimeDiff = 0;
        double averageProcessDurationDiff = 0;

        Console.WriteLine($"Comparing {Comparisons.Count} groups");
        foreach (var c in Comparisons) 
        {
            double percentageDiff(double b, double c) => (c - b) / b * 100;
            var baseline = c.Value.Baseline;
            var comparand = c.Value.Comparand;

            averageTotalPauseTimeDiffPercentage += percentageDiff(baseline.Stats.GetGCPauseTimePercentage(), comparand.Stats.GetGCPauseTimePercentage());
            averageTotalAllocDiffPercentage += percentageDiff(baseline.Stats.TotalAllocatedMB, comparand.Stats.TotalAllocatedMB);
            averageMeanHeapSizeBeforeDiffPercentage += percentageDiff(baseline.Stats.MeanSizePeakMB, comparand.Stats.MeanSizePeakMB);
            averageGen0PauseTimeDiffPercentage += percentageDiff(baseline.Generations[0].TotalPauseTimeMSec, comparand.Generations[0].TotalPauseTimeMSec);
            averageGen1PauseTimeDiffPercentage += percentageDiff(baseline.Generations[1].TotalPauseTimeMSec, comparand.Generations[1].TotalPauseTimeMSec);
            averageGen2PauseTimeDiffPercentage += percentageDiff(baseline.Gen2Blocking.Sum(gc => gc.PauseDurationMSec), comparand.Gen2Blocking.Sum(gc => gc.PauseDurationMSec));
            averageFirstLastTimeDiff = percentageDiff(baseline.GCs.Last().PauseStartRelativeMSec - baseline.GCs.First().PauseStartRelativeMSec, comparand.GCs.Last().PauseStartRelativeMSec - comparand.GCs.First().PauseStartRelativeMSec);
            averageProcessDurationDiff = percentageDiff(baseline.Stats.ProcessDuration, comparand.Stats.ProcessDuration);
        }

        DiffStatistics = new DiffStatistics
        {
            AverageTotalPauseTimeDiffPercentage = averageTotalPauseTimeDiffPercentage / Comparisons.Count,
            AverageTotalAllocDiffPercentage = averageTotalAllocDiffPercentage / Comparisons.Count,
            AverageMeanHeapSizeDiffPercentage = averageMeanHeapSizeBeforeDiffPercentage / Comparisons.Count,
            AverageGen0PauseTimeDiffPercentage = averageGen0PauseTimeDiffPercentage / Comparisons.Count,
            AverageGen1PauseTimeDiffPercentage = averageGen1PauseTimeDiffPercentage / Comparisons.Count,
            AverageGen2BlockingPauseTimeDiffPercentage = averageGen2PauseTimeDiffPercentage / Comparisons.Count,
            AverageFirstLastTimeDiff = averageFirstLastTimeDiff / Comparisons.Count,
            AverageProcessDurationDiff = averageProcessDurationDiff / Comparisons.Count
        };

        foreach (var g in Comparisons)
        {
            g.Value.Baseline.Compare(new [] { g.Value.Comparand });
        }
    }

    public static Dictionary<int, Comparison> GetCombinedGroups(Dictionary<string, Analyzer> gcTraceData)
    {
        var result = new Dictionary<int, Comparison>();

        foreach (var entry in gcTraceData)
        {
            string fileName = Path.GetFileNameWithoutExtension(entry.Key);
            int index = fileName.LastIndexOf('-') + 1;
            int num = fileName[index] - '0';

            if (!result.ContainsKey(num))
            {
                result[num] = new Comparison();
            }

            if (fileName.Contains("base"))
            {
                result[num].Baseline = entry.Value.AllGCProcessData.First().Value.First();
            }
            else
            {
                result[num].Comparand = entry.Value.AllGCProcessData.First().Value.First();
            }
        }
        
        return result;
    }

    public void WriteComparisons(string outputBasePath)
    {
        foreach (var g in Comparisons)
        {
            g.Value.Baseline.Compare(new [] { g.Value.Comparand }).ToMarkdown($"{outputBasePath}/{g.Key}.md");
        }
    }

    public Dictionary<int, Comparison> Comparisons { get; } 
    public DiffStatistics DiffStatistics { get; }
}

In [11]:
ComparisonManager comparisonManagerNoEnv = new(basePath_NoEnv);
Console.WriteLine("Statistics for NoEnv:");
comparisonManagerNoEnv.WriteComparisons("./Comparisons/NoEnv");
comparisonManagerNoEnv.DiffStatistics.Display();

ComparisonManager comparisonManager8Heaps = new(basePath_8HeapCount);
Console.WriteLine("Statistics for 8 Heaps:");
comparisonManager8Heaps.DiffStatistics.Display();
comparisonManagerNoEnv.WriteComparisons("./Comparisons/8Heaps");

Comparing 5 groups
Statistics for NoEnv:


AverageTotalPauseTimeDiffPercentage,-4.387051258197316
AverageMeanHeapSizeDiffPercentage,-0.45839963360696634
AverageTotalAllocDiffPercentage,-0.4779480023024282
AverageGen0PauseTimeDiffPercentage,-17.6589055158066
AverageGen1PauseTimeDiffPercentage,-5.512819465612246
AverageGen2BlockingPauseTimeDiffPercentage,-9.253508169997465


Comparing 5 groups
Statistics for 8 Heaps:


AverageTotalPauseTimeDiffPercentage,-4.696338106922325
AverageMeanHeapSizeDiffPercentage,-0.39978026652744847
AverageTotalAllocDiffPercentage,-0.4605404283990467
AverageGen0PauseTimeDiffPercentage,-21.19999769106091
AverageGen1PauseTimeDiffPercentage,-11.142173902682746
AverageGen2BlockingPauseTimeDiffPercentage,-8.155615900097544


In [7]:
Console.WriteLine($"Current Process ID: {System.Diagnostics.Process.GetCurrentProcess().Id}");

#!about

Current Process ID: 36928


.NET Interactive© 2020-2024 Microsoft CorporationVersion: 1.0.611002+2d9aef81050acdc8d17055b8ed4066284636e31eLibrary version: 1.0.0-beta.25110.2+2d9aef81050acdc8d17055b8ed4066284636e31eBuild date: 2025-03-01T03:33:33.6354597Zhttps://github.com/dotnet/interactive
